# Bài 5 trang 26: Gom cụm dữ liệu MNIST bằng k-Means

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Bài làm dùng tập train MNIST từ Keras, gom ảnh thành 10 cụm và so sánh cụm với nhãn chữ số thật. Nhãn thật chỉ được dùng khi đánh giá, không được dùng để huấn luyện k-Means.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, accuracy_score, confusion_matrix
from scipy.optimize import linear_sum_assignment

HO_TEN, MSSV = "Lê Nguyễn Anh Du", "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

in_thong_tin()
print("Đã nhập các thư viện cần thiết.")

## 1) Đọc dữ liệu MNIST và xem ảnh mẫu

In [ ]:
in_thong_tin()
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print("Kích thước train_images:", train_images.shape)
print("Kích thước train_labels:", train_labels.shape)
print("Kích thước test_images:", test_images.shape)
print("Kích thước test_labels:", test_labels.shape)
print("Số ảnh theo từng nhãn trong tập train:")
print(pd.Series(train_labels).value_counts().sort_index().to_string())
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for digit, ax in enumerate(axes):
    index = np.where(train_labels == digit)[0][0]
    ax.imshow(train_images[index], cmap="gray")
    ax.set_title(str(digit))
    ax.axis("off")
plt.suptitle("Một ảnh mẫu cho mỗi chữ số")
plt.tight_layout()
plt.show()

## 2) Chuẩn hóa và biểu diễn ảnh dưới dạng vector

Mỗi ảnh 28 × 28 pixel được làm phẳng thành 784 giá trị, rồi chia cho 255. PCA giảm xuống 50 chiều để k-Means trên toàn bộ 60.000 ảnh train chạy thuận tiện hơn. PCA và k-Means không dùng nhãn lớp.

In [ ]:
in_thong_tin()
X = train_images.reshape(len(train_images), -1).astype("float32") / 255.0
pca = PCA(n_components=50, svd_solver="randomized", random_state=42)
X_pca = pca.fit_transform(X)
print("Kích thước dữ liệu sau làm phẳng:", X.shape)
print("Kích thước dữ liệu sau PCA:", X_pca.shape)
print(f"Tỷ lệ phương sai được giữ lại: {pca.explained_variance_ratio_.sum():.2%}")

## 3) Gom cụm bằng k-Means với 10 cụm tương ứng 10 chữ số

In [ ]:
in_thong_tin()
kmeans = KMeans(n_clusters=10, init="k-means++", n_init=3, max_iter=300, random_state=42)
cluster_labels = kmeans.fit_predict(X_pca)
print("Kích thước kết quả gom cụm:", cluster_labels.shape)
print("Số ảnh trong từng cụm:")
print(pd.Series(cluster_labels).value_counts().sort_index().to_string())
print(f"Inertia: {kmeans.inertia_:.2f}")

## 4) So sánh cụm với nhãn thật

Mã số cụm 0–9 do k-Means tạo ra không mang nghĩa chữ số 0–9. Bảng chéo cho biết mỗi cụm chứa bao nhiêu ảnh của từng chữ số thật.

In [ ]:
in_thong_tin()
comparison = pd.crosstab(
    pd.Series(cluster_labels, name="Cụm k-Means"),
    pd.Series(train_labels, name="Nhãn thật")
).reindex(index=range(10), columns=range(10), fill_value=0)
display(comparison)
plt.figure(figsize=(10, 7))
sns.heatmap(comparison, annot=True, fmt="d", cmap="Blues")
plt.title("Số ảnh của từng chữ số trong mỗi cụm")
plt.xlabel("Nhãn chữ số thật")
plt.ylabel("Cụm k-Means")
plt.tight_layout()
plt.show()

## 5) Đánh giá mức khớp giữa cụm và nhãn lớp

ARI và NMI so sánh phân nhóm mà không phụ thuộc tên cụm. Để tính tỷ lệ trùng nhãn, dùng thuật toán Hungarian ghép một cụm với một chữ số sao cho tổng số ảnh đúng là lớn nhất. Đây là chỉ số mô tả trên cùng tập dữ liệu đã gom cụm.

In [ ]:
in_thong_tin()
row_indices, col_indices = linear_sum_assignment(-comparison.to_numpy())
cluster_to_digit = {int(cluster): int(digit) for cluster, digit in zip(row_indices, col_indices)}
mapped_predictions = np.array([cluster_to_digit[int(cluster)] for cluster in cluster_labels])
print("Ghép cụm → chữ số:", cluster_to_digit)
print(f"Adjusted Rand Index (ARI): {adjusted_rand_score(train_labels, cluster_labels):.4f}")
print(f"Normalized Mutual Information (NMI): {normalized_mutual_info_score(train_labels, cluster_labels):.4f}")
print(f"Tỷ lệ trùng nhãn sau ghép tối ưu: {accuracy_score(train_labels, mapped_predictions):.4f}")
print("Số ảnh có chữ số thật trùng chữ số được ghép:", int(np.sum(train_labels == mapped_predictions)))

In [ ]:
in_thong_tin()
cm = confusion_matrix(train_labels, mapped_predictions, labels=np.arange(10))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=np.arange(10), yticklabels=np.arange(10))
plt.title("Nhãn thật và cụm sau khi ghép với chữ số")
plt.xlabel("Chữ số gán cho cụm")
plt.ylabel("Chữ số thật")
plt.tight_layout()
plt.show()

## 6) Minh họa vị trí các cụm trong hai thành phần PCA đầu tiên

In [ ]:
in_thong_tin()
rng = np.random.default_rng(42)
plot_indices = rng.choice(len(X_pca), size=3000, replace=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca[plot_indices, 0], X_pca[plot_indices, 1], c=train_labels[plot_indices], cmap="tab10", s=8, alpha=0.7)
axes[0].set_title("Màu theo nhãn chữ số thật")
axes[1].scatter(X_pca[plot_indices, 0], X_pca[plot_indices, 1], c=cluster_labels[plot_indices], cmap="tab10", s=8, alpha=0.7)
axes[1].set_title("Màu theo cụm k-Means")
for ax in axes:
    ax.set_xlabel("Thành phần PCA 1")
    ax.set_ylabel("Thành phần PCA 2")
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
in_thong_tin()
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for cluster_id, ax in enumerate(axes):
    member_indices = np.where(cluster_labels == cluster_id)[0]
    distances = np.linalg.norm(X_pca[member_indices] - kmeans.cluster_centers_[cluster_id], axis=1)
    representative_index = member_indices[np.argmin(distances)]
    ax.imshow(train_images[representative_index], cmap="gray")
    ax.set_title(f"Cụm {cluster_id}\n→ {cluster_to_digit[cluster_id]}")
    ax.axis("off")
plt.suptitle("Ảnh đại diện gần tâm cụm nhất")
plt.tight_layout()
plt.show()